In [0]:

# 1. Retrieve the source_name passed from the Databricks Job
dbutils.widgets.text("source_system_id", "1")
source_id = dbutils.widgets.get("source_system_id").strip()

print(f"Fetching configuration for source: {source_id}...")

# 2. Query the Unity Catalog configuration table for Data Dictionary config
query = f"""
    SELECT 
        source_id,
        source_name,
        source_type,
        host,
        port,
        database_name,
        driver_class,
        secret_scope,
        secret_key_credentials,
        source_id,
        Data_Dictionary_Enable,
        landing_volume_path
    FROM migration_x_catalog.pfl_x_schema.config_source_system 
    WHERE source_id = int('{source_id}') and is_active = 1;
"""
config_df = spark.sql(query).collect()

if not config_df:
    print(f"WARNING: No configuration found for source '{source_name}'. Exiting.")
    dbutils.jobs.taskValues.set(key="run_data_dict", value="false")
    dbutils.notebook.exit(f"No config found for source: {source_name}")

source_dict = config_df[0]

# 3. Extract Data Dictionary parameters
dd_enabled = int(source_dict["Data_Dictionary_Enable"] or 0)
source_name = str(source_dict["source_name"]).strip() if source_dict["source_name"] else ""

# Push task values for downstream workflow tasks
run_data_dict = "true" if dd_enabled == 1 else "false"
print(f"Data_Dictionary_Enable = {dd_enabled} | run_data_dict = {run_data_dict}")

In [0]:
# DATA DICTIONARY EXECUTION FLOW
# Only runs when Data_Dictionary_Enable = 1

if dd_enabled != 1:
    print("Data Dictionary is DISABLED for this source. Skipping.")
    dbutils.notebook.exit("SKIPPED: Data Dictionary not enabled.")

print(f"Data Dictionary is ENABLED. Starting flow for source: {source_name}")


dbutils.jobs.taskValues.set(key="dd_enabled", value=dd_enabled)
dbutils.jobs.taskValues.set(key="source_name", value=source_name)